# DichVideo Batch ASR to SRT

Upload many video/audio files, run faster-whisper on Colab GPU, and download only `.srt` results.

In [ ]:
!pip -q install -U faster-whisper


In [ ]:
# Use the Whisper model already uploaded to Google Drive. This cell does not download from HuggingFace.
# Upload the full model folder to: MyDrive/models/faster-whisper-large-v3
USE_DRIVE_MODEL = True
MODEL_NAME = 'large-v3'

if USE_DRIVE_MODEL:
    from google.colab import drive
    from pathlib import Path
    import shutil

    drive.mount('/content/drive')
    drive_model_dir = Path(f'/content/drive/MyDrive/models/faster-whisper-{MODEL_NAME}')
    local_model_dir = Path(f'/content/models/faster-whisper-{MODEL_NAME}')

    if not (drive_model_dir / 'model.bin').exists():
        raise FileNotFoundError(
            'Model not found in Google Drive. Upload the downloaded folder so this file exists: '
            f'{drive_model_dir / "model.bin"}'
        )

    print(f'Using model from Drive: {drive_model_dir}')
    if not (local_model_dir / 'model.bin').exists():
        print(f'Copying model from Drive to fast local disk: {local_model_dir}')
        local_model_dir.parent.mkdir(parents=True, exist_ok=True)
        if local_model_dir.exists():
            shutil.rmtree(local_model_dir)
        shutil.copytree(drive_model_dir, local_model_dir)

    MODEL_PATH = str(local_model_dir)
else:
    MODEL_PATH = MODEL_NAME

print('MODEL_PATH =', MODEL_PATH)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

UPLOAD_DIR = Path('/content/dichvideo_asr_uploads')
OUTPUT_DIR = Path('/content/dichvideo_asr_srt')
shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(data)

print('Uploaded files:')
for path in sorted(UPLOAD_DIR.iterdir()):
    print('-', path.name)

In [ ]:
%%writefile /content/colab_batch_asr_to_srt.py
from __future__ import annotations

import argparse
import logging
import re
import shutil
import subprocess
import time
from pathlib import Path

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi', '.m4v', '.mp3', '.wav', '.m4a'}

def main() -> None:
    parser = argparse.ArgumentParser(description='Batch ASR videos/audios to SRT with faster-whisper.')
    parser.add_argument('--input-dir', required=True)
    parser.add_argument('--output-dir', required=True)
    parser.add_argument('--model', default='small')
    parser.add_argument('--language', default=None)
    parser.add_argument('--device', default='cuda', choices=['cuda', 'cpu', 'auto'])
    parser.add_argument('--compute-type', default='float16')
    parser.add_argument('--beam-size', type=int, default=8)
    parser.add_argument('--word-timestamps', action='store_true')
    parser.add_argument('--vad-min-silence-ms', type=int, default=500)
    parser.add_argument('--speech-pad-ms', type=int, default=200)
    parser.add_argument('--max-subtitle-seconds', type=float, default=0)
    args = parser.parse_args()
    input_dir = Path(args.input_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    logger = _setup_logger(output_dir / 'batch_asr_to_srt.log')
    files = sorted(path for path in input_dir.iterdir() if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS)
    if not files:
        raise RuntimeError(f'No supported video/audio files found in {input_dir}')
    logger.info('Found %s input file(s): %s', len(files), ', '.join(path.name for path in files))
    from faster_whisper import WhisperModel
    load_started = time.monotonic()
    logger.info('Loading model=%s device=%s compute_type=%s (first run can spend minutes downloading weights)', args.model, args.device, args.compute_type)
    model = WhisperModel(args.model, device=args.device, compute_type=args.compute_type)
    logger.info('Model loaded in %.1fs', time.monotonic() - load_started)
    for index, media_path in enumerate(files, start=1):
        duration = _probe_duration(media_path)
        size_mb = media_path.stat().st_size / (1024 * 1024)
        logger.info('Transcribing %s/%s: %s (%.1f MB%s)', index, len(files), media_path, size_mb, f', {duration:.1f}s' if duration is not None else '')
        transcribe_started = time.monotonic()
        vad_parameters = {'min_silence_duration_ms': args.vad_min_silence_ms, 'speech_pad_ms': args.speech_pad_ms}
        segments_iter, info = model.transcribe(
            str(media_path),
            language=args.language or None,
            vad_filter=True,
            vad_parameters=vad_parameters,
            beam_size=args.beam_size,
            word_timestamps=args.word_timestamps,
        )
        logger.info('Decoder returned segment iterator for %s; consuming segments...', media_path.name)
        srt_segments = []
        for seg in segments_iter:
            text = ' '.join(seg.text.strip().split())
            if text:
                start = float(seg.start)
                end = float(seg.end)
                words = getattr(seg, 'words', None) or []
                if words:
                    word_starts = [float(w.start) for w in words if getattr(w, 'start', None) is not None]
                    word_ends = [float(w.end) for w in words if getattr(w, 'end', None) is not None]
                    if word_starts:
                        start = min(word_starts)
                    if word_ends:
                        end = max(word_ends)
                if args.max_subtitle_seconds and end - start > args.max_subtitle_seconds:
                    end = start + args.max_subtitle_seconds
                srt_segments.append({'start': start, 'end': max(start + 0.2, end), 'text': text})
                if len(srt_segments) == 1 or len(srt_segments) % 25 == 0:
                    logger.info('Progress %s: %s subtitle segment(s), audio position %.1fs', media_path.name, len(srt_segments), float(seg.end))
        out = output_dir / f'{_safe_stem(media_path)}.srt'
        out.write_text(_to_srt(srt_segments), encoding='utf-8')
        logger.info('Wrote %s segments to %s language=%s probability=%.4f', len(srt_segments), out, getattr(info, 'language', None), getattr(info, 'language_probability', 0.0))
        logger.info('Finished %s in %.1fs', media_path.name, time.monotonic() - transcribe_started)
    shutil.make_archive('/content/asr_srt_results', 'zip', output_dir)

def _probe_duration(path: Path) -> float | None:
    try:
        result = subprocess.run([
            'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1', str(path),
        ], check=True, capture_output=True, text=True)
        return float(result.stdout.strip())
    except Exception:
        return None

def _safe_stem(path: Path) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', path.stem).strip('._') or 'media'

def _to_srt(segments: list[dict]) -> str:
    return '\n'.join(f"{i}\n{_ts(seg['start'])} --> {_ts(seg['end'])}\n{seg['text']}\n" for i, seg in enumerate(segments, start=1))

def _ts(seconds: float) -> str:
    total_ms = max(0, int(round(seconds * 1000)))
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def _setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger('dichvideo_batch_asr_to_srt')
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s')
    fh = logging.FileHandler(log_path, encoding='utf-8')
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    sh = logging.StreamHandler()
    sh.setFormatter(formatter)
    logger.addHandler(sh)
    return logger

if __name__ == '__main__':
    main()


In [ ]:
MODEL = MODEL_PATH       # Drive model copied to local Colab disk, or model name if USE_DRIVE_MODEL=False
LANGUAGE = 'zh'         # '' = auto, or 'zh', 'en', 'vi', ...
DEVICE = 'cuda'
COMPUTE_TYPE = 'float16'
BEAM_SIZE = 8
WORD_TIMESTAMPS = True
VAD_MIN_SILENCE_MS = 350
SPEECH_PAD_MS = 80
MAX_SUBTITLE_SECONDS = 4.0  # 0 = no cap

!python -u /content/colab_batch_asr_to_srt.py \
  --input-dir "$UPLOAD_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --model "$MODEL" \
  --language "$LANGUAGE" \
  --device "$DEVICE" \
  --compute-type "$COMPUTE_TYPE" \
  --beam-size "$BEAM_SIZE" \
  --vad-min-silence-ms "$VAD_MIN_SILENCE_MS" \
  --speech-pad-ms "$SPEECH_PAD_MS" \
  --max-subtitle-seconds "$MAX_SUBTITLE_SECONDS" \
  --word-timestamps

In [ ]:
from google.colab import files
files.download('/content/asr_srt_results.zip')